# Lekcija 01 - Uvod u AI agente

Dobrodošli na prvu lekciju u tečaju **AI agenata za početnike**!

**AI agent** je program koji koristi velik jezični model (LLM) kao svoj mehanizam rezoniranja i može poduzimati *akcije* u stvarnom svijetu — pozivati API-je, ispitivati baze podataka ili izvršavati kod — kako bi ostvario cilj u ime korisnika.

U ovom bilježniku izgradit ćete svog prvog agenta: **Putničkog agenta** koji preporučuje destinacije za odmor. Usput ćete naučiti kako:

1. Povezati se s Azure AI Foundry Agent Service koristeći **Microsoft Agent Framework**.
2. Dati agentu **alat** — običnu Python funkciju koju može pozvati.
3. Pokrenuti agenta i pregledati njegov odgovor.
4. Streamati odgovor agenta token po token.


## Postavljanje

Prije pokretanja ovog bilježnika, pobrinite se da imate:

1. **Azure AI Foundry projekt** s implementiranim chat modelom (npr. `gpt-4o-mini`).
2. **Prijavljeni ste s Azure CLI** — pokrenite `az login` u vašem terminalu.
3. **Postavljene potrebne varijable okruženja:**
   - `AZURE_AI_PROJECT_ENDPOINT` — krajnju točku vašeg Azure AI Foundry projekta.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ime vašeg implementiranog modela.

Ćelija ispod instalira Python pakete koje trebate.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Kreiranje vašeg prvog agenta

Agentu su potrebne dvije stvari:

- **Upute** koje mu govore *tko je* i *kako se ponašati* (sistemski prompt).
- **Alati** — Python funkcije označene s `@tool` koje agent može pozvati za dohvaćanje informacija ili izvođenje radnji.

Ispod definiramo jednostavan alat koji vraća popis popularnih destinacija za odmor. Agent će koristiti ovaj alat kada korisnik zatraži preporuke za putovanja.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming odgovori

Za interaktivnije iskustvo možete **streamati** odgovor agenta. Umjesto da čekate cijeli odgovor, agent isporučuje tekst u dijelovima dok ih generira. Ovo je posebno korisno u chat sučeljima gdje želite prikazati izlaz u stvarnom vremenu.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Sažetak

U ovoj lekciji naučili ste kako:

- **Stvoriti providera** koji se povezuje na Azure AI Foundry Agent Service putem `AzureAIProjectAgentProvider`.
- **Definirati alat** koristeći dekorator `@tool` kako bi agent mogao pozivati vaše Python funkcije.
- **Pokrenuti agenta** s porukom korisnika i ispisati njegov odgovor.
- **Strimirati odgovore** za prikaz u stvarnom vremenu.

U sljedećoj lekciji detaljnije ćemo istražiti agentne okvire i naučiti kako agentima dati moćnije alate i mogućnosti višekorakog rezoniranja.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Odricanje od odgovornosti**:
Ovaj dokument je preveden korištenjem AI usluge prevođenja [Co-op Translator](https://github.com/Azure/co-op-translator). Iako nastojimo postići točnost, molimo imajte na umu da automatski prijevodi mogu sadržavati pogreške ili netočnosti. Izvorni dokument na izvornom jeziku treba se smatrati službenim izvorom. Za kritične informacije preporučuje se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazuma ili krive interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
